# M2 — Data acquisition and quality

Download adjusted USD prices and monthly uranium/CPI inputs, preserve exact inputs, and inspect quality before approving the research universe. First acquisition requires internet access. Repeated runs reuse a checksum-validated local cache. The end date is exclusive and fixed in configuration.

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


## 1. Acquire and record provenance

`auto_adjust=True` adjusts Yahoo OHLC for vendor corporate actions; raw share volume remains unadjusted. This is a practical total-return proxy with vendor limitations. Each security's currency is checked against USD. FRED inputs are latest revised series, not a vintage database. Raw data are local and ignored by Git; provider redistribution rights are not assumed.

In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers)
print("Retrieved:", source_metadata["retrieved_at_utc"])
print("Cache:", source_metadata["cache_path"])
print("Price rows:", len(prices), "Macro rows:", len(macro))
print("Macro sources:", source_metadata["macro_sources"])
prices.head()


## 2. Inspect schema and identity

One row represents one security on one trading date. Duplicate keys would double count observations. Prices must be finite and positive; zero volume is possible and should be investigated rather than silently replaced. Compare actual provider metadata with the candidate identity record.

In [ ]:
print(prices.dtypes)
print("Duplicate keys:", prices.duplicated(["ticker", "date"]).sum())
print("Missing values:", prices.isna().sum().to_dict())
identity = pd.DataFrame(source_metadata["instruments"]).T
print(identity[["currency", "exchangeName", "longName"]].to_string())
assert prices.currency.eq("USD").all()


## 3. Validate history, freshness and gaps

A missing benchmark session before a company's listing is different from a gap during its listed history. The report retains first/last dates and missing-session counts for inspection. Large daily moves may be real or a corporate-action issue and require review. Liquidity eligibility is trailing and date-specific; a current liquid stock is not assumed liquid throughout history.

In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers, offline=True)
prices, quality, targets, table = research_tables(cfg, prices, macro, universe)
display(quality)
print("Corporate-action events:")
display(prices.loc[(prices.dividends != 0) | (prices.stock_splits != 0), ["date", "ticker", "dividends", "stock_splits"]].tail(12))
print("Months eligible per candidate:")
display(table.groupby("ticker").eligible.agg(["sum", "count"]))


## 4. Save approved clean outputs

The initial candidates were chosen before viewing performance from the verified URNM/URNJ/NUKZ snapshots. URA, HURA and NLR are documented candidate-source omissions. This is a current-constituent historical study, not reconstructed historical ETF membership.

In [ ]:
processed = ROOT / "data/processed"
processed.mkdir(exist_ok=True)
prices.to_csv(processed / "clean_prices.csv", index=False)
macro.to_csv(processed / "clean_macro.csv", index=False)
quality.to_csv(processed / "data_quality.csv", index=False)
assert quality.minimum_history_met.all()
assert prices.date.max() < pd.Timestamp(cfg["end"])
print("Checkpoint passed: input schema, freshness, unique keys and minimum histories.")


## Written-result checkpoint

Record which candidates meet the history rule, which have short histories, any extreme-return or corporate-action warnings, and the actual common analysis periods. Explain why missing pre-listing sessions are not filled. Continue only after checking the warnings and source limitations.